# Fixed random weighting → Final Classifier — multi-seed + multi-selection

This notebook is a **no-Rater control experiment** with the same downstream
classifier protocol, seeds, optimizer, validation-selection rules, metrics,
checkpoints, plots, and Grad-CAM structure as the frozen-Rater notebook.

For each final-classifier seed, every training sample receives one fixed random
weight:

\[
w_i \sim \mathrm{Uniform}(0,1)
\]

exactly once before training begins. The same \(w_i\) is reused for that sample
for all 50 epochs. The random weight does **not** depend on label, group,
correctness, loss, or any Rater output.

The classifier is optimized with:

\[
L_{\mathrm{train}}
=
\sum_i w_i\,CE_i.
\]

There is **no FeatureRater / Rater model anywhere in this experiment**.

In [1]:
# ============================================================
# 1. Mount Drive + imports
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
from collections import defaultdict
import copy
import csv
import json
import math
import os
import random
import re
import subprocess
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/evidential_alignment_colab"
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    DEVICE,
)

Mounted at /content/drive
Device: cpu


In [2]:
# ============================================================
# 2. EXPERIMENT CONFIGURATION
# ============================================================

EMBEDDING_CACHE_ROOT = Path(
    "/content/drive/MyDrive/evidential_alignment_colab/"
    "embeddings_waterbirds_erm/waterbirds/"
    "resnet50_waterbirds_erm_1055b1e8de29/"
    "resolution_224"
)

EXPECTED_ERM_FINGERPRINT = (
    "1055b1e8de29"
)

# ---------------- Final-classifier seeds ----------------

FINAL_CLASSIFIER_SEEDS = [0, 1, 2, 3, 4, 42, 69]

# Recommended:
# keep exactly the same classifier train/selection split across optimizer seeds.
VARY_DATA_SPLIT_WITH_SEED = True
DATA_SPLIT_SEED = 0
SPLIT_VAL = 0.6


# ---------------- Fixed random sample weighting ----------------
#
# Every training sample gets exactly ONE random weight at the beginning of
# its seed run. That weight is reused for every epoch.
#
# The random weights are Uniform[0,1). They DO NOT depend on label, group,
# correctness, loss, or any Rater output.
# ------------------------------------------------------------

# WEIGHT_MODE = "random":   one Uniform[low, high) weight per sample (control).
# WEIGHT_MODE = "constant": every sample gets CONSTANT_WEIGHT = uniform-weight
#   retraining on the same embeddings, splits, seeds and selection rules.
#   0.5 matches the mean of Uniform[0,1) and of the sigmoid rater weights, so
#   the total weight mass (effective step size) is comparable.
WEIGHT_MODE = "random"
CONSTANT_WEIGHT = 0.5

RANDOM_WEIGHT_LOW = 0.0
RANDOM_WEIGHT_HIGH = 1.0

# Independent deterministic RNG stream for the sample weights.
# For classifier seed s:
#   random-weight seed = RANDOM_WEIGHT_SEED_OFFSET + s
RANDOM_WEIGHT_SEED_OFFSET = 100000

# True:
#   seed 0/1/2 get different, but reproducible, sample-weight assignments.
# False:
#   every classifier seed uses the exact same random sample-weight vector.
VARY_RANDOM_WEIGHTS_WITH_CLASSIFIER_SEED = True

AUX_CE_COEFF = 0.0

assert WEIGHT_MODE in {"random", "constant"}

WEIGHTING_POLICY = (
    "fixed_random_uniform_per_sample"
    if WEIGHT_MODE == "random"
    else "constant_weight_uniform_retraining"
)


# ---------------- Final classifier ----------------

BATCH_SIZE = 128
NUM_WORKERS = 2
FINAL_EPOCHS = 50
FINAL_LR = 1e-2
FINAL_MOMENTUM = 0.9
FINAL_WEIGHT_DECAY = 1e-4


# ---------------- Selection rules ----------------

SELECTION_RULES = {
    "val_wga": {
        "field": "wga",
        "mode": "max",
        "label": "Validation WGA",
    },
    "val_accuracy": {
        "field": "accuracy",
        "mode": "max",
        "label": "Validation overall accuracy",
    },
    "val_worst_class_accuracy": {
        "field": "worst_class_accuracy",
        "mode": "max",
        "label": "Validation worst-class accuracy",
    },
    "val_mean_class_accuracy": {
        "field": "mean_class_accuracy",
        "mode": "max",
        "label": "Validation mean-class accuracy",
    },
    "val_mean_group_accuracy": {
        "field": "mean_group_accuracy",
        "mode": "max",
        "label": "Validation mean-group accuracy",
    },
    "val_loss": {
        "field": "loss",
        "mode": "min",
        "label": "Validation mean CE",
    },
    "val_worst_group_loss": {
        "field": "worst_group_loss",
        "mode": "min",
        "label": "Validation worst-group mean CE",
    },
    "val_worst_class_loss": {
        "field": "worst_class_loss",
        "mode": "min",
        "label": "Validation worst-class mean CE",
    },
}

PRIMARY_SELECTION_RULE = "val_wga"

ORACLE_RULES = {
    "oracle_test_wga": {
        "field": "wga",
        "mode": "max",
        "label": "Oracle test WGA",
    },
    "oracle_test_accuracy": {
        "field": "accuracy",
        "mode": "max",
        "label": "Oracle test overall accuracy",
    },
}



# ---------------- Diagnostics ----------------

EVAL_EVERY = 1
PLOT_EVERY = 1
CHECKPOINT_EVERY = 0

ENABLE_GRADCAM = True
GRADCAM_SPLIT = "test"
GRADCAM_SAMPLES_PER_GROUP = 1
GRADCAM_SELECTION_RULES = ["val_wga"]

# ---------------- Grad-CAM image-space paths ----------------

REPO_DIR = Path(
    "/content/evidential_alignment"
)

DATA_ROOT = Path(
    "/content/datasets"
)

WB_DIR = (
    DATA_ROOT
    / "waterbird_complete95_forest2water2"
)

WB_ARCHIVE = (
    DRIVE_ROOT
    / "archives"
    / "waterbird_complete95_forest2water2.tar.gz"
)

DEFAULT_ERM_MODEL = (
    DRIVE_ROOT
    / "erm_base_models"
    / "waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt"
)

if not DEFAULT_ERM_MODEL.exists():
    DEFAULT_ERM_MODEL = (
        DRIVE_ROOT
        / "erm_base_models"
        / "erm_waterbirds_resnet50_train_128B_100E_seed0_pretrained"
        / "best_val_acc_model.pt"
    )


# ---------------- One folder for the whole run ----------------

RUNS_ROOT = (
    DRIVE_ROOT / "fixed_random_weight_final_multiseed_multiselection"
)


RUN_NAME = (
    (
        f"random_uniform_{RANDOM_WEIGHT_LOW}_{RANDOM_WEIGHT_HIGH}_"
        if WEIGHT_MODE == "random"
        else f"constant_{CONSTANT_WEIGHT}_"
    )
    + f"lr_{FINAL_LR}_splitval_{SPLIT_VAL}_"
    + f"{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)


EXPERIMENT_ROOT = (
    RUNS_ROOT
    / RUN_NAME
)

EXPERIMENT_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

AGGREGATE_PLOTS_ROOT = (
    EXPERIMENT_ROOT
    / "aggregate_plots"
)

AGGREGATE_PLOTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert len(FINAL_CLASSIFIER_SEEDS) >= 1
assert len(set(FINAL_CLASSIFIER_SEEDS)) == len(FINAL_CLASSIFIER_SEEDS)
assert PRIMARY_SELECTION_RULE in SELECTION_RULES
assert all(
    rule in SELECTION_RULES
    for rule in GRADCAM_SELECTION_RULES
)
assert EMBEDDING_CACHE_ROOT.exists(), EMBEDDING_CACHE_ROOT

print(
    "Weighting policy:",
    WEIGHTING_POLICY,
)
print(
    "Final-classifier seeds:",
    FINAL_CLASSIFIER_SEEDS,
)
print(
    "Embedding cache:",
    EMBEDDING_CACHE_ROOT,
)
print(
    "Experiment root:",
    EXPERIMENT_ROOT,
)

Weighting policy: fixed_random_uniform_per_sample
Final-classifier seeds: [0, 1, 2, 3, 4, 42, 69]
Embedding cache: /content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm/waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224
Experiment root: /content/drive/MyDrive/evidential_alignment_colab/fixed_random_weight_final_multiseed_multiselection/random_uniform_0.0_1.0_lr_0.01_splitval_0.6_20261013_140627


In [3]:
# ============================================================
# 3. Reproducibility + artifact helpers
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_value(value):
    if torch.is_tensor(value):
        return json_value(value.detach().cpu().tolist())
    if isinstance(value, np.ndarray):
        return json_value(value.tolist())
    if isinstance(value, np.generic):
        return json_value(value.item())
    if isinstance(value, dict):
        return {str(k): json_value(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [json_value(v) for v in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return str(value)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + f".tmp.{os.getpid()}")
    with open(tmp, "w", encoding="utf-8") as fout:
        json.dump(
            json_value(payload),
            fout,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        fout.write("\n")
    os.replace(tmp, path)


def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + f".tmp.{os.getpid()}")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def append_csv_rows(path, rows):
    if not rows:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists() and path.stat().st_size > 0
    with open(path, "a", newline="", encoding="utf-8") as fout:
        writer = csv.DictWriter(
            fout,
            fieldnames=list(rows[0].keys()),
        )
        if not exists:
            writer.writeheader()
        writer.writerows(rows)


def write_csv(path, rows):
    if not rows:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as fout:
        writer = csv.DictWriter(
            fout,
            fieldnames=list(rows[0].keys()),
        )
        writer.writeheader()
        writer.writerows(rows)


def cpu_state_dict(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else []
        ),
    }


def make_logger(path):
    path = Path(path)

    def _log(message=""):
        message = str(message)
        print(message, flush=True)
        with open(path, "a", encoding="utf-8") as fout:
            fout.write(message + "\n")

    return _log


def mean_std(values):
    values = np.asarray(values, dtype=np.float64)
    return (
        float(values.mean()),
        float(values.std(ddof=1)) if len(values) > 1 else 0.0,
    )



root_config = {
    "stage": "final_classifier_multiseed",
    "final_classifier_seeds": FINAL_CLASSIFIER_SEEDS,
    "vary_data_split_with_seed": VARY_DATA_SPLIT_WITH_SEED,
    "data_split_seed": DATA_SPLIT_SEED,
    "split_val": SPLIT_VAL,
    "embedding_cache_root": str(EMBEDDING_CACHE_ROOT),
    "expected_erm_fingerprint": EXPECTED_ERM_FINGERPRINT,

    "weighting_policy": WEIGHTING_POLICY,
    "random_weight_low": RANDOM_WEIGHT_LOW,
    "random_weight_high": RANDOM_WEIGHT_HIGH,
    "random_weight_seed_offset": RANDOM_WEIGHT_SEED_OFFSET,
    "vary_random_weights_with_classifier_seed": (
        VARY_RANDOM_WEIGHTS_WITH_CLASSIFIER_SEED
    ),

    "batch_size": BATCH_SIZE,
    "epochs": FINAL_EPOCHS,
    "lr": FINAL_LR,
    "momentum": FINAL_MOMENTUM,
    "weight_decay": FINAL_WEIGHT_DECAY,
    "aux_ce_coeff": AUX_CE_COEFF,
    "selection_rules": SELECTION_RULES,
    "primary_selection_rule": PRIMARY_SELECTION_RULE,
    "oracle_rules": ORACLE_RULES,
    "checkpoint_every": CHECKPOINT_EVERY,
}

write_json(
    EXPERIMENT_ROOT
    / "config.json",
    root_config,
)

In [4]:
# ============================================================
# 4. Embedding datasets + fixed/varying split support
# ============================================================

class IndexedEmbeddingDataset(Dataset):
    def __init__(
        self,
        embeddings,
        labels,
        groups,
        attrs,
        split_name,
        source_indices=None,
    ):
        self.embeddings = embeddings.float()
        self.labels = labels.long()
        self.groups = groups.long()
        self.attrs = attrs.long()
        self.split_name = split_name
        self.source_indices = (
            torch.arange(len(labels))
            if source_indices is None
            else torch.as_tensor(source_indices).long()
        )

    def __len__(self):
        return int(self.labels.shape[0])

    def __getitem__(self, idx):
        return (
            idx,
            self.embeddings[idx],
            self.labels[idx],
            self.groups[idx],
            self.attrs[idx],
        )


def split_token(value):
    return str(value).replace(".", "p")


def embedding_split_path(split_name, split_seed):
    if split_name in {"train_no_aug", "val", "test"}:
        return EMBEDDING_CACHE_ROOT / f"{split_name}.pt"

    if split_name in {"val_subset1", "val_subset2"}:
        return EMBEDDING_CACHE_ROOT / (
            f"{split_name}_"
            f"splittrain_1p0_"
            f"splitval_{split_token(SPLIT_VAL)}_"
            f"seed_{split_seed}.pt"
        )

    raise ValueError(split_name)


def ensure_val_subsets(split_seed):
    subset1_path = embedding_split_path(
        "val_subset1",
        split_seed,
    )
    subset2_path = embedding_split_path(
        "val_subset2",
        split_seed,
    )

    if subset1_path.exists() and subset2_path.exists():
        return

    full_val_path = embedding_split_path(
        "val",
        split_seed,
    )

    payload = torch.load(
        full_val_path,
        map_location="cpu",
        weights_only=False,
    )

    n = int(payload["labels"].shape[0])

    rng = np.random.RandomState(split_seed)
    indices = np.arange(n)
    rng.shuffle(indices)

    n_subset1 = int(n * SPLIT_VAL)

    split_indices = {
        "val_subset1": torch.as_tensor(
            indices[:n_subset1],
            dtype=torch.long,
        ),
        "val_subset2": torch.as_tensor(
            indices[n_subset1:],
            dtype=torch.long,
        ),
    }

    for split_name, idx in split_indices.items():
        target = embedding_split_path(
            split_name,
            split_seed,
        )

        subset = {
            "embeddings": payload["embeddings"][idx].clone(),
            "labels": payload["labels"][idx].clone(),
            "groups": payload["groups"][idx].clone(),
            "attrs": payload["attrs"][idx].clone(),
            "source_indices": idx.clone(),
            "cache_metadata": dict(
                payload.get("cache_metadata", {})
            ),
        }

        subset["cache_metadata"].update({
            "split": split_name,
            "split_train": 1.0,
            "split_val": float(SPLIT_VAL),
            "seed": int(split_seed),
            "derived_from_full_val_cache": True,
        })

        torch.save(
            subset,
            target,
        )


def load_embedding_split(split_name, split_seed):
    path = embedding_split_path(
        split_name,
        split_seed,
    )

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    embeddings = payload["embeddings"]

    if embeddings.ndim != 2 or embeddings.shape[1] != 2048:
        raise ValueError(
            f"{split_name}: unexpected embedding shape "
            f"{tuple(embeddings.shape)}"
        )

    metadata = payload.get(
        "cache_metadata",
        {},
    )

    representation = metadata.get(
        "representation_source"
    )

    if (
        representation
        and representation != "waterbirds_erm_checkpoint"
    ):
        raise RuntimeError(
            f"{split_name} is not a Waterbirds-ERM cache."
        )

    fingerprint = (
        metadata.get("erm_checkpoint_fingerprint")
        or metadata.get("erm_fingerprint")
    )

    if (
        fingerprint
        and fingerprint != EXPECTED_ERM_FINGERPRINT
    ):
        raise RuntimeError(
            f"{split_name}: embedding fingerprint mismatch: "
            f"{fingerprint}"
        )

    return IndexedEmbeddingDataset(
        payload["embeddings"],
        payload["labels"],
        payload["groups"],
        payload["attrs"],
        split_name=split_name,
        source_indices=payload.get("source_indices"),
    )


def load_datasets_for_split_seed(split_seed):
    ensure_val_subsets(split_seed)

    datasets = {
        "train": load_embedding_split(
            "val_subset1",
            split_seed,
        ),
        "val": load_embedding_split(
            "val_subset2",
            split_seed,
        ),
        "test": load_embedding_split(
            "test",
            split_seed,
        ),
    }

    split_names = {
        "train": "val_subset1",
        "val": "val_subset2",
        "test": "test",
    }

    return datasets, split_names

In [5]:
# ============================================================
# 5. Fixed random per-sample weighting policy
# ============================================================
#
# NO Rater is loaded or used in this experiment.
#
# Each training sample is assigned one Uniform[0,1) weight before training.
# The assignment is deterministic for the seed and remains unchanged for all
# epochs. Labels, groups, correctness, and losses do not affect the weights.
# ============================================================

WEIGHTING_SIGNATURE = {
    "policy": WEIGHTING_POLICY,
    "uses_rater": False,
    "uses_group_annotations_for_training_weights": False,
    "weight_mode": WEIGHT_MODE,
    "constant_weight": (
        CONSTANT_WEIGHT if WEIGHT_MODE == "constant" else None
    ),
    "distribution": (
        "Uniform[low, high)" if WEIGHT_MODE == "random" else "constant"
    ),
    "low": RANDOM_WEIGHT_LOW,
    "high": RANDOM_WEIGHT_HIGH,
    "random_weight_seed_offset": RANDOM_WEIGHT_SEED_OFFSET,
    "vary_random_weights_with_classifier_seed": (
        VARY_RANDOM_WEIGHTS_WITH_CLASSIFIER_SEED
    ),
    "weights_constant_across_training": True,
}

root_config[
    "weighting_signature"
] = WEIGHTING_SIGNATURE

write_json(
    EXPERIMENT_ROOT
    / "config.json",
    root_config,
)

print(
    json.dumps(
        WEIGHTING_SIGNATURE,
        indent=2,
    )
)

{
  "policy": "fixed_random_uniform_per_sample",
  "uses_rater": false,
  "uses_group_annotations_for_training_weights": false,
  "distribution": "Uniform[low, high)",
  "low": 0.0,
  "high": 1.0,
  "random_weight_seed_offset": 100000,
  "vary_random_weights_with_classifier_seed": true,
  "weights_constant_across_training": true
}


In [6]:
# ============================================================
# 6. Final-classifier metrics, selection rules, and plots
# ============================================================

class FinalLinearClassifier(nn.Module):
    def __init__(self, input_dim=2048, num_classes=2):
        super().__init__()
        self.linear = nn.Linear(
            input_dim,
            num_classes,
        )

    def forward(self, z):
        return self.linear(z)


@torch.no_grad()
def evaluate_classifier(model, dataset):
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_n = 0

    group_correct = {}
    group_total = {}
    group_loss_sum = {}

    class_correct = {}
    class_total = {}
    class_loss_sum = {}

    for _, z, y, g, _ in loader:
        z = z.to(
            DEVICE,
            non_blocking=True,
        )
        y = y.to(
            DEVICE,
            non_blocking=True,
        )
        g = g.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(z)

        per_loss = F.cross_entropy(
            logits,
            y,
            reduction="none",
        )

        pred = logits.argmax(dim=1)

        total_loss += float(
            per_loss.sum().item()
        )
        total_correct += int(
            pred.eq(y).sum().item()
        )
        total_n += int(
            y.numel()
        )

        for group_tensor in torch.unique(g):
            gid = int(
                group_tensor.item()
            )
            mask = g == group_tensor

            group_correct[gid] = (
                group_correct.get(gid, 0)
                + int(
                    pred[mask]
                    .eq(y[mask])
                    .sum()
                    .item()
                )
            )

            group_total[gid] = (
                group_total.get(gid, 0)
                + int(mask.sum().item())
            )

            group_loss_sum[gid] = (
                group_loss_sum.get(
                    gid,
                    0.0,
                )
                + float(
                    per_loss[mask]
                    .sum()
                    .item()
                )
            )

        for class_tensor in torch.unique(y):
            cid = int(
                class_tensor.item()
            )
            mask = y == class_tensor

            class_correct[cid] = (
                class_correct.get(cid, 0)
                + int(
                    pred[mask]
                    .eq(y[mask])
                    .sum()
                    .item()
                )
            )

            class_total[cid] = (
                class_total.get(cid, 0)
                + int(mask.sum().item())
            )

            class_loss_sum[cid] = (
                class_loss_sum.get(
                    cid,
                    0.0,
                )
                + float(
                    per_loss[mask]
                    .sum()
                    .item()
                )
            )

    if total_n == 0:
        raise RuntimeError(
            "Cannot evaluate an empty dataset."
        )

    group_accuracy = {
        gid: (
            group_correct[gid]
            / group_total[gid]
        )
        for gid in sorted(group_total)
    }

    group_loss = {
        gid: (
            group_loss_sum[gid]
            / group_total[gid]
        )
        for gid in sorted(group_total)
    }

    class_accuracy = {
        cid: (
            class_correct[cid]
            / class_total[cid]
        )
        for cid in sorted(class_total)
    }

    class_loss = {
        cid: (
            class_loss_sum[cid]
            / class_total[cid]
        )
        for cid in sorted(class_total)
    }

    return {
        "n_samples": total_n,
        "loss": (
            total_loss
            / total_n
        ),
        "accuracy": (
            total_correct
            / total_n
        ),
        "group_accuracy": group_accuracy,
        "group_loss": group_loss,
        "group_count": group_total,
        "wga": min(
            group_accuracy.values()
        ),
        "mean_group_accuracy": float(
            np.mean(
                list(
                    group_accuracy.values()
                )
            )
        ),
        "worst_group_loss": max(
            group_loss.values()
        ),
        "class_accuracy": class_accuracy,
        "class_loss": class_loss,
        "class_count": class_total,
        "worst_class_accuracy": min(
            class_accuracy.values()
        ),
        "mean_class_accuracy": float(
            np.mean(
                list(
                    class_accuracy.values()
                )
            )
        ),
        "worst_class_loss": max(
            class_loss.values()
        ),
        "mean_class_loss": float(
            np.mean(
                list(
                    class_loss.values()
                )
            )
        ),
    }


def is_better(candidate, best, mode):
    if best is None:
        return True
    if mode == "max":
        return candidate > best
    if mode == "min":
        return candidate < best
    raise ValueError(mode)


def save_five_panel_histogram(
    values,
    groups,
    path,
    xlabel,
    title,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )
    groups = np.asarray(
        groups,
        dtype=np.int64,
    )

    finite = values[
        np.isfinite(values)
    ]

    if len(finite) == 0:
        return

    vmin = float(
        finite.min()
    )
    vmax = float(
        finite.max()
    )

    if np.isclose(
        vmin,
        vmax,
    ):
        eps = max(
            abs(vmin) * 0.05,
            1e-6,
        )
        bins = np.linspace(
            vmin - eps,
            vmax + eps,
            20,
        )
    else:
        bins = np.linspace(
            vmin,
            vmax,
            41,
        )

    fig = plt.figure(
        figsize=(16, 12)
    )

    grid = fig.add_gridspec(
        4,
        2,
        width_ratios=[
            1.0,
            2.15,
        ],
        hspace=0.50,
        wspace=0.28,
    )

    for row, gid in enumerate(
        [0, 1, 2, 3]
    ):
        ax = fig.add_subplot(
            grid[row, 0]
        )

        mask = groups == gid

        if bool(mask.any()):
            ax.hist(
                values[mask],
                bins=bins,
                density=True,
                alpha=0.80,
            )

        ax.set_xlim(
            bins[0],
            bins[-1],
        )
        ax.set_ylabel(
            "Density"
        )
        ax.set_title(
            f"Group {gid} "
            f"(n={int(mask.sum())})"
        )
        ax.grid(
            True,
            linestyle=":",
            alpha=0.25,
        )

        if row == 3:
            ax.set_xlabel(
                xlabel
            )

    ax_all = fig.add_subplot(
        grid[:, 1]
    )

    for gid in [0, 1, 2, 3]:
        mask = groups == gid

        if not bool(mask.any()):
            continue

        ax_all.hist(
            values[mask],
            bins=bins,
            density=True,
            alpha=0.38,
            label=(
                f"Group {gid} "
                f"(n={int(mask.sum())})"
            ),
        )

    ax_all.set_xlim(
        bins[0],
        bins[-1],
    )
    ax_all.set_xlabel(
        xlabel
    )
    ax_all.set_ylabel(
        "Density"
    )
    ax_all.set_title(
        "All groups overlaid"
    )
    ax_all.grid(
        True,
        linestyle=":",
        alpha=0.25,
    )
    ax_all.legend(
        fontsize=9
    )

    fig.suptitle(
        title,
        fontsize=15,
    )
    fig.subplots_adjust(
        top=0.94
    )

    fig.savefig(
        path,
        dpi=160,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_trajectory_plots(
    metric_rows,
    group_rows,
    class_rows,
    plots_root,
):
    trajectory = [
        row
        for row in metric_rows
        if row["phase"] == "trajectory"
    ]

    overall_fields = [
        (
            "accuracy",
            "Accuracy",
        ),
        (
            "wga",
            "Worst-group accuracy",
        ),
        (
            "loss",
            "Mean CE loss",
        ),
        (
            "worst_group_loss",
            "Worst-group mean CE loss",
        ),
    ]

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(13, 9),
    )

    for ax, (
        field,
        label,
    ) in zip(
        axes.flat,
        overall_fields,
    ):
        for role in [
            "train",
            "val",
            "test",
        ]:
            rows = sorted(
                [
                    row
                    for row in trajectory
                    if row["role"] == role
                ],
                key=lambda row: row["epoch"],
            )

            ax.plot(
                [
                    row["epoch"]
                    for row in rows
                ],
                [
                    row[field]
                    for row in rows
                ],
                marker=".",
                label=role,
            )

        ax.set_xlabel(
            "Epoch"
        )
        ax.set_ylabel(
            label
        )
        ax.grid(
            True,
            alpha=0.25,
        )
        ax.legend()

    fig.suptitle(
        "Final classifier: train / val / test"
    )
    fig.tight_layout()

    fig.savefig(
        plots_root
        / "overall_metrics_over_time.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.close(fig)

    classaware_fields = [
        (
            "worst_class_accuracy",
            "Worst-class accuracy",
        ),
        (
            "mean_class_accuracy",
            "Mean-class accuracy",
        ),
        (
            "mean_group_accuracy",
            "Mean-group accuracy",
        ),
        (
            "worst_class_loss",
            "Worst-class mean CE",
        ),
    ]

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(13, 9),
    )

    for ax, (
        field,
        label,
    ) in zip(
        axes.flat,
        classaware_fields,
    ):
        for role in [
            "train",
            "val",
            "test",
        ]:
            rows = sorted(
                [
                    row
                    for row in trajectory
                    if row["role"] == role
                ],
                key=lambda row: row["epoch"],
            )

            ax.plot(
                [
                    row["epoch"]
                    for row in rows
                ],
                [
                    row[field]
                    for row in rows
                ],
                marker=".",
                label=role,
            )

        ax.set_xlabel(
            "Epoch"
        )
        ax.set_ylabel(
            label
        )
        ax.grid(
            True,
            alpha=0.25,
        )
        ax.legend()

    fig.suptitle(
        "Final classifier: class/balanced metrics over time"
    )
    fig.tight_layout()

    fig.savefig(
        plots_root
        / "class_balanced_metrics_over_time.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.close(fig)

    for role in [
        "train",
        "val",
        "test",
    ]:
        role_group_rows = [
            row
            for row in group_rows
            if (
                row["phase"] == "trajectory"
                and row["role"] == role
            )
        ]

        fig, axes = plt.subplots(
            1,
            2,
            figsize=(14, 5),
        )

        for ax, field, ylabel in [
            (
                axes[0],
                "accuracy",
                "Group accuracy",
            ),
            (
                axes[1],
                "loss",
                "Group mean CE",
            ),
        ]:
            for gid in sorted(
                {
                    row["group"]
                    for row in role_group_rows
                }
            ):
                rows = sorted(
                    [
                        row
                        for row in role_group_rows
                        if row["group"] == gid
                    ],
                    key=lambda row: row["epoch"],
                )

                ax.plot(
                    [
                        row["epoch"]
                        for row in rows
                    ],
                    [
                        row[field]
                        for row in rows
                    ],
                    marker=".",
                    label=f"Group {gid}",
                )

            ax.set_xlabel(
                "Epoch"
            )
            ax.set_ylabel(
                ylabel
            )
            ax.grid(
                True,
                alpha=0.25,
            )
            ax.legend(
                fontsize=8
            )

        fig.suptitle(
            f"{role}: per-group trajectory"
        )
        fig.tight_layout()

        fig.savefig(
            plots_root
            / f"group_metrics_{role}_over_time.png",
            dpi=160,
            bbox_inches="tight",
        )
        plt.close(fig)

        role_class_rows = [
            row
            for row in class_rows
            if (
                row["phase"] == "trajectory"
                and row["role"] == role
            )
        ]

        fig, axes = plt.subplots(
            1,
            2,
            figsize=(12, 5),
        )

        for ax, field, ylabel in [
            (
                axes[0],
                "accuracy",
                "Class accuracy",
            ),
            (
                axes[1],
                "loss",
                "Class mean CE",
            ),
        ]:
            for cid in sorted(
                {
                    row["class_id"]
                    for row in role_class_rows
                }
            ):
                rows = sorted(
                    [
                        row
                        for row in role_class_rows
                        if row["class_id"] == cid
                    ],
                    key=lambda row: row["epoch"],
                )

                ax.plot(
                    [
                        row["epoch"]
                        for row in rows
                    ],
                    [
                        row[field]
                        for row in rows
                    ],
                    marker=".",
                    label=f"Class {cid}",
                )

            ax.set_xlabel(
                "Epoch"
            )
            ax.set_ylabel(
                ylabel
            )
            ax.grid(
                True,
                alpha=0.25,
            )
            ax.legend(
                fontsize=8
            )

        fig.suptitle(
            f"{role}: per-class trajectory"
        )
        fig.tight_layout()

        fig.savefig(
            plots_root
            / f"class_metrics_{role}_over_time.png",
            dpi=160,
            bbox_inches="tight",
        )
        plt.close(fig)


def save_selection_comparison_plot(
    selection_results,
    oracle_results,
    path,
):
    names = [
        row["selection_rule"]
        for row in selection_results
    ]

    test_accuracy = np.asarray(
        [
            row["test_accuracy"]
            for row in selection_results
        ],
        dtype=np.float64,
    )

    test_wga = np.asarray(
        [
            row["test_wga"]
            for row in selection_results
        ],
        dtype=np.float64,
    )

    oracle_wga = next(
        (
            row["test_wga"]
            for row in oracle_results
            if row["oracle_rule"] == "oracle_test_wga"
        ),
        None,
    )

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(13, 10),
    )

    axes[0].bar(
        names,
        test_accuracy,
    )
    axes[0].set_ylabel(
        "Test overall accuracy"
    )
    axes[0].tick_params(
        axis="x",
        labelrotation=35,
    )
    axes[0].grid(
        True,
        axis="y",
        alpha=0.25,
    )

    axes[1].bar(
        names,
        test_wga,
    )
    axes[1].set_ylabel(
        "Test WGA"
    )
    axes[1].tick_params(
        axis="x",
        labelrotation=35,
    )
    axes[1].grid(
        True,
        axis="y",
        alpha=0.25,
    )

    if oracle_wga is not None:
        axes[1].axhline(
            oracle_wga,
            linestyle="--",
            label="Oracle test WGA",
        )
        axes[1].legend()

    fig.suptitle(
        "Test outcome under each validation selection rule"
    )
    fig.tight_layout()

    fig.savefig(
        path,
        dpi=160,
        bbox_inches="tight",
    )
    plt.close(fig)

In [7]:
# ============================================================
# 7. Fixed training weights — random Uniform[0,1) per sample
# ============================================================

def build_fixed_training_weights(
    dataset,
    seed,
):
    random_weight_seed = int(
        RANDOM_WEIGHT_SEED_OFFSET
    )

    if VARY_RANDOM_WEIGHTS_WITH_CLASSIFIER_SEED:
        random_weight_seed += int(
            seed
        )

    generator = (
        torch.Generator(
            device="cpu"
        )
        .manual_seed(
            random_weight_seed
        )
    )

    weights = (
        RANDOM_WEIGHT_LOW
        + (
            RANDOM_WEIGHT_HIGH
            - RANDOM_WEIGHT_LOW
        )
        * torch.rand(
            len(dataset),
            generator=generator,
            dtype=torch.float32,
        )
    )

    if WEIGHT_MODE == "constant":
        # Uniform-weight retraining: identical weight for every sample.
        weights = torch.full(
            (len(dataset),),
            float(CONSTANT_WEIGHT),
            dtype=torch.float32,
        )

    metadata = {
        "policy": WEIGHTING_POLICY,
        "weight_mode": WEIGHT_MODE,
        "constant_weight": (
            float(CONSTANT_WEIGHT)
            if WEIGHT_MODE == "constant"
            else None
        ),
        "distribution": (
            "Uniform[low, high)"
            if WEIGHT_MODE == "random"
            else "constant"
        ),
        "low": float(
            RANDOM_WEIGHT_LOW
        ),
        "high": float(
            RANDOM_WEIGHT_HIGH
        ),
        "random_weight_seed": (
            random_weight_seed
        ),
        "vary_random_weights_with_classifier_seed": (
            VARY_RANDOM_WEIGHTS_WITH_CLASSIFIER_SEED
        ),
        "weights_constant_across_training": True,
        "uses_group_annotations_for_training_weights": False,
    }

    return weights, metadata


@torch.no_grad()
def compute_training_weights(
    batch_indices,
    logits,
    y,
    fixed_training_weights,
):
    indices_cpu = (
        batch_indices
        .detach()
        .cpu()
        .long()
    )

    weights = (
        fixed_training_weights[
            indices_cpu
        ].to(
            logits.device,
            dtype=logits.dtype,
        )
    )

    pred = logits.argmax(
        dim=1
    )

    correct_mask = pred.eq(
        y
    )

    wrong_mask = (
        ~correct_mask
    )

    return (
        weights,
        correct_mask,
        wrong_mask,
        None,
    )

def weight_stat_rows(
    seed,
    epoch,
    weights,
    groups,
    correctness,
):
    weights = np.asarray(
        weights,
        dtype=np.float64,
    )
    groups = np.asarray(
        groups,
        dtype=np.int64,
    )
    correctness = (
        np.asarray(correctness)
        > 0.5
    )

    masks = {
        "all": np.ones(
            len(weights),
            dtype=bool,
        ),
        "correct": correctness,
        "wrong": ~correctness,
    }

    for gid in sorted(
        np.unique(groups)
    ):
        masks[
            f"group_{gid}"
        ] = (
            groups == gid
        )

    rows = []

    for subset, mask in masks.items():
        values = weights[mask]

        if len(values) == 0:
            continue

        rows.append({
            "seed": int(seed),
            "epoch": int(epoch),
            "subset": subset,
            "n": int(len(values)),
            "mean": float(
                values.mean()
            ),
            "std": float(
                values.std()
            ),
            "min": float(
                values.min()
            ),
            "q25": float(
                np.quantile(
                    values,
                    0.25,
                )
            ),
            "median": float(
                np.median(values)
            ),
            "q75": float(
                np.quantile(
                    values,
                    0.75,
                )
            ),
            "max": float(
                values.max()
            ),
            "mass_share": float(
                values.sum()
                / max(
                    weights.sum(),
                    1e-12,
                )
            ),
        })

    return rows

In [8]:
# ============================================================
# 8. Train one final-classifier seed and save all selections
# ============================================================

def run_one_seed(seed):
    seed = int(seed)

    split_seed = (
        seed
        if VARY_DATA_SPLIT_WITH_SEED
        else DATA_SPLIT_SEED
    )

    seed_root = (
        EXPERIMENT_ROOT
        / f"seed_{seed}"
    )
    plots_root = (
        seed_root
        / "plots"
    )
    data_root = (
        plots_root
        / "data"
    )
    checkpoint_root = (
        seed_root
        / "checkpoints"
    )
    selection_checkpoint_root = (
        checkpoint_root
        / "by_selection"
    )
    selected_model_root = (
        seed_root
        / "selected_models"
    )

    for path in [
        seed_root,
        plots_root,
        data_root,
        checkpoint_root,
        selection_checkpoint_root,
        selected_model_root,
    ]:
        path.mkdir(
            parents=True,
            exist_ok=True,
        )

    log = make_logger(
        seed_root
        / "log.txt"
    )

    seed_start = time.monotonic()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    log("=" * 100)
    log(
        f"FINAL CLASSIFIER SEED {seed}"
    )
    log("=" * 100)
    log(
        f"data split seed = {split_seed}"
    )
    log(
        f"weighting policy = {WEIGHTING_POLICY}"
    )

    datasets_by_role, split_names = (
        load_datasets_for_split_seed(
            split_seed
        )
    )

    for role, dataset in datasets_by_role.items():
        log(
            f"{role}={split_names[role]}: "
            f"n={len(dataset)}"
        )

    fixed_training_weights, weighting_metadata = (
        build_fixed_training_weights(
            datasets_by_role[
                "train"
            ],
            seed,
        )
    )

    if len(
        fixed_training_weights
    ) != len(
        datasets_by_role[
            "train"
        ]
    ):
        raise RuntimeError(
            "Fixed training-weight vector has the wrong length."
        )

    # Save the exact weight assigned to every training sample.
    atomic_torch_save(
        {
            "weights": (
                fixed_training_weights
                .detach()
                .cpu()
            ),
            "labels": (
                datasets_by_role[
                    "train"
                ].labels.detach()
                .cpu()
            ),
            "groups": (
                datasets_by_role[
                    "train"
                ].groups.detach()
                .cpu()
            ),
            "source_indices": (
                datasets_by_role[
                    "train"
                ].source_indices.detach()
                .cpu()
            ),
            "metadata": (
                weighting_metadata
            ),
        },
        seed_root
        / "fixed_training_weights.pt",
    )

    sample_weight_rows = []

    for sample_index in range(
        len(
            datasets_by_role[
                "train"
            ]
        )
    ):
        sample_weight_rows.append(
            {
                "sample_index": (
                    sample_index
                ),
                "source_index": int(
                    datasets_by_role[
                        "train"
                    ].source_indices[
                        sample_index
                    ].item()
                ),
                "label": int(
                    datasets_by_role[
                        "train"
                    ].labels[
                        sample_index
                    ].item()
                ),
                "group": int(
                    datasets_by_role[
                        "train"
                    ].groups[
                        sample_index
                    ].item()
                ),
                "weight": float(
                    fixed_training_weights[
                        sample_index
                    ].item()
                ),
            }
        )

    write_csv(
        seed_root
        / "fixed_training_weights.csv",
        sample_weight_rows,
    )

    save_five_panel_histogram(
        fixed_training_weights.numpy(),
        datasets_by_role[
            "train"
        ].groups.numpy(),
        plots_root
        / "fixed_training_weights_by_group.png",
        "Fixed training weight",
        (
            f"Fixed sample weights | "
            f"seed {seed} | "
            f"{WEIGHTING_POLICY}"
        ),
    )

    seed_config = {
        **root_config,
        "seed": seed,
        "split_seed": split_seed,
        "weighting_metadata": (
            weighting_metadata
        ),
        "split_names": (
            split_names
        ),
        "split_sizes": {
            role: len(dataset)
            for role, dataset
            in datasets_by_role.items()
        },
    }

    write_json(
        seed_root
        / "config.json",
        seed_config,
    )

    set_seed(seed)

    model = FinalLinearClassifier(
        2048,
        2,
    ).to(
        DEVICE
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=FINAL_LR,
        momentum=FINAL_MOMENTUM,
        weight_decay=FINAL_WEIGHT_DECAY,
    )

    train_loader = DataLoader(
        datasets_by_role["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    metrics_csv = (
        data_root
        / "metrics.csv"
    )
    group_metrics_csv = (
        data_root
        / "group_metrics.csv"
    )
    class_metrics_csv = (
        data_root
        / "class_metrics.csv"
    )
    history_csv = (
        data_root
        / "history.csv"
    )
    weight_stats_csv = (
        data_root
        / "weight_stats.csv"
    )

    metric_rows = []
    group_rows = []
    class_rows = []

    def record_metrics(
        epoch,
        role,
        metrics,
        phase,
        selection_rule="",
    ):
        row = {
            "seed": seed,
            "phase": phase,
            "selection_rule": selection_rule,
            "epoch": int(epoch),
            "role": role,
            "split": (
                split_names[role]
            ),
            "n_samples": (
                metrics["n_samples"]
            ),
            "accuracy": (
                metrics["accuracy"]
            ),
            "wga": (
                metrics["wga"]
            ),
            "mean_group_accuracy": (
                metrics[
                    "mean_group_accuracy"
                ]
            ),
            "loss": (
                metrics["loss"]
            ),
            "worst_group_loss": (
                metrics[
                    "worst_group_loss"
                ]
            ),
            "worst_class_accuracy": (
                metrics[
                    "worst_class_accuracy"
                ]
            ),
            "mean_class_accuracy": (
                metrics[
                    "mean_class_accuracy"
                ]
            ),
            "worst_class_loss": (
                metrics[
                    "worst_class_loss"
                ]
            ),
            "mean_class_loss": (
                metrics[
                    "mean_class_loss"
                ]
            ),
        }

        metric_rows.append(row)
        append_csv_rows(
            metrics_csv,
            [row],
        )

        local_group_rows = []

        for gid in sorted(
            metrics["group_count"]
        ):
            local_group_rows.append({
                "seed": seed,
                "phase": phase,
                "selection_rule": (
                    selection_rule
                ),
                "epoch": int(epoch),
                "role": role,
                "split": (
                    split_names[role]
                ),
                "group": int(gid),
                "n_samples": int(
                    metrics[
                        "group_count"
                    ][gid]
                ),
                "accuracy": (
                    metrics[
                        "group_accuracy"
                    ][gid]
                ),
                "loss": (
                    metrics[
                        "group_loss"
                    ][gid]
                ),
            })

        group_rows.extend(
            local_group_rows
        )
        append_csv_rows(
            group_metrics_csv,
            local_group_rows,
        )

        local_class_rows = []

        for cid in sorted(
            metrics["class_count"]
        ):
            local_class_rows.append({
                "seed": seed,
                "phase": phase,
                "selection_rule": (
                    selection_rule
                ),
                "epoch": int(epoch),
                "role": role,
                "split": (
                    split_names[role]
                ),
                "class_id": int(cid),
                "n_samples": int(
                    metrics[
                        "class_count"
                    ][cid]
                ),
                "accuracy": (
                    metrics[
                        "class_accuracy"
                    ][cid]
                ),
                "loss": (
                    metrics[
                        "class_loss"
                    ][cid]
                ),
            })

        class_rows.extend(
            local_class_rows
        )
        append_csv_rows(
            class_metrics_csv,
            local_class_rows,
        )

    def checkpoint_payload(
        epoch,
        metrics_by_role,
        kind,
    ):
        return {
            "schema_version": 3,
            "stage": (
                "final_classifier"
            ),
            "checkpoint_kind": kind,
            "seed": seed,
            "split_seed": split_seed,
            "epoch": int(epoch),
            "model_sd": (
                cpu_state_dict(model)
            ),
            "optimizer_sd": copy.deepcopy(
                optimizer.state_dict()
            ),
            "rng_state": rng_state(),
            "metrics": metrics_by_role,
            "config": seed_config,
            "weighting_policy": (
                WEIGHTING_POLICY
            ),
            "weighting_metadata": (
                weighting_metadata
            ),
            "embedding_cache_root": str(
                EMBEDDING_CACHE_ROOT
            ),
            "erm_fingerprint": (
                EXPECTED_ERM_FINGERPRINT
            ),
        }

    # Fresh-model epoch-0 diagnostics.
    initial_metrics = {
        role: evaluate_classifier(
            model,
            dataset,
        )
        for role, dataset
        in datasets_by_role.items()
    }

    for role, metrics in initial_metrics.items():
        record_metrics(
            0,
            role,
            metrics,
            phase="trajectory",
        )

    selection_trackers = {
        rule_name: {
            "best_value": None,
            "best_epoch": None,
        }
        for rule_name
        in SELECTION_RULES
    }

    oracle_trackers = {
        rule_name: {
            "best_value": None,
            "best_epoch": None,
            "test_metrics": None,
        }
        for rule_name
        in ORACLE_RULES
    }

    for epoch in range(
        1,
        FINAL_EPOCHS + 1,
    ):
        model.train()

        weighted_objective_sum = 0.0
        unweighted_ce_sum = 0.0
        seen = 0

        epoch_weights = []
        epoch_groups = []
        epoch_correctness = []

        for (
            batch_indices,
            z,
            y,
            g,
            _,
        ) in train_loader:
            z = z.to(
                DEVICE,
                non_blocking=True,
            )
            y = y.to(
                DEVICE,
                non_blocking=True,
            )
            g = g.to(
                DEVICE,
                non_blocking=True,
            )

            logits = model(z)

            (
                weights,
                correct_mask,
                _,
                _,
            ) = compute_training_weights(
                batch_indices=batch_indices,
                logits=logits,
                y=y,
                fixed_training_weights=(
                    fixed_training_weights
                ),
            )

            per_sample_ce = (
                F.cross_entropy(
                    logits,
                    y,
                    reduction="none",
                )
            )

            weighted_ce = (
                per_sample_ce
                * weights
            ).sum()

            ordinary_ce = (
                per_sample_ce.mean()
            )

            objective = (
                weighted_ce
                + AUX_CE_COEFF
                * ordinary_ce
            )

            optimizer.zero_grad()
            objective.backward()
            optimizer.step()

            weighted_objective_sum += float(
                weighted_ce.item()
            )
            unweighted_ce_sum += float(
                per_sample_ce.sum()
                .item()
            )
            seen += int(
                y.numel()
            )

            epoch_weights.append(
                weights.detach().cpu()
            )
            epoch_groups.append(
                g.detach().cpu()
            )
            epoch_correctness.append(
                correct_mask.float()
                .detach()
                .cpu()
            )

        weights_np = torch.cat(
            epoch_weights
        ).numpy()
        groups_np = torch.cat(
            epoch_groups
        ).numpy()
        correctness_np = torch.cat(
            epoch_correctness
        ).numpy()

        append_csv_rows(
            weight_stats_csv,
            weight_stat_rows(
                seed,
                epoch,
                weights_np,
                groups_np,
                correctness_np,
            ),
        )

        if (
            epoch == 1
            or epoch % PLOT_EVERY == 0
            or epoch == FINAL_EPOCHS
        ):
            weight_plot_root = (
                plots_root
                / "weight_histograms"
            )
            weight_plot_root.mkdir(
                parents=True,
                exist_ok=True,
            )

            save_five_panel_histogram(
                weights_np,
                groups_np,
                weight_plot_root
                / (
                    f"epoch_{epoch:03d}_"
                    "weights_by_group.png"
                ),
                "Actual training weight",
                (
                    f"Training weights | "
                    f"seed {seed} | epoch {epoch}"
                ),
            )

        metrics_by_role = {
            role: evaluate_classifier(
                model,
                dataset,
            )
            for role, dataset
            in datasets_by_role.items()
        }

        for role, metrics in metrics_by_role.items():
            record_metrics(
                epoch,
                role,
                metrics,
                phase="trajectory",
            )

        val_metrics = (
            metrics_by_role["val"]
        )

        # Save a different best checkpoint for every validation rule.
        for rule_name, rule in SELECTION_RULES.items():
            current_value = float(
                val_metrics[
                    rule["field"]
                ]
            )

            tracker = (
                selection_trackers[
                    rule_name
                ]
            )

            if is_better(
                current_value,
                tracker["best_value"],
                rule["mode"],
            ):
                tracker[
                    "best_value"
                ] = current_value
                tracker[
                    "best_epoch"
                ] = epoch

                checkpoint = checkpoint_payload(
                    epoch,
                    metrics_by_role,
                    (
                        "validation_selected_"
                        + rule_name
                    ),
                )

                checkpoint.update({
                    "selection_rule": (
                        rule_name
                    ),
                    "selection_field": (
                        rule["field"]
                    ),
                    "selection_mode": (
                        rule["mode"]
                    ),
                    "selection_value": (
                        current_value
                    ),
                })

                atomic_torch_save(
                    checkpoint,
                    selection_checkpoint_root
                    / f"{rule_name}.pt",
                )

        # Diagnostic test oracles.
        test_metrics = (
            metrics_by_role["test"]
        )

        for (
            oracle_name,
            oracle_rule,
        ) in ORACLE_RULES.items():
            current_value = float(
                test_metrics[
                    oracle_rule["field"]
                ]
            )

            tracker = (
                oracle_trackers[
                    oracle_name
                ]
            )

            if is_better(
                current_value,
                tracker["best_value"],
                oracle_rule["mode"],
            ):
                tracker[
                    "best_value"
                ] = current_value
                tracker[
                    "best_epoch"
                ] = epoch
                tracker[
                    "test_metrics"
                ] = copy.deepcopy(
                    test_metrics
                )

                oracle_checkpoint = (
                    checkpoint_payload(
                        epoch,
                        metrics_by_role,
                        (
                            "TEST_ORACLE_"
                            "DIAGNOSTIC_ONLY_"
                            + oracle_name
                        ),
                    )
                )

                oracle_checkpoint.update({
                    "oracle_rule": (
                        oracle_name
                    ),
                    "oracle_field": (
                        oracle_rule["field"]
                    ),
                    "oracle_value": (
                        current_value
                    ),
                    "diagnostic_only": True,
                })

                atomic_torch_save(
                    oracle_checkpoint,
                    checkpoint_root
                    / f"{oracle_name}.pt",
                )

        latest = checkpoint_payload(
            epoch,
            metrics_by_role,
            "latest",
        )

        latest[
            "selection_trackers"
        ] = selection_trackers
        latest[
            "oracle_trackers"
        ] = oracle_trackers

        atomic_torch_save(
            latest,
            checkpoint_root
            / "latest.pt",
        )

        if (
            CHECKPOINT_EVERY > 0
            and epoch
            % CHECKPOINT_EVERY == 0
        ):
            atomic_torch_save(
                checkpoint_payload(
                    epoch,
                    metrics_by_role,
                    "periodic",
                ),
                checkpoint_root
                / f"epoch_{epoch:03d}.pt",
            )

        history_row = {
            "seed": seed,
            "epoch": epoch,
            "weighted_objective_sum": (
                weighted_objective_sum
            ),
            "unweighted_mean_ce_seen_during_training": (
                unweighted_ce_sum
                / max(
                    seen,
                    1,
                )
            ),
            "weight_mean": float(
                weights_np.mean()
            ),
            "weight_std": float(
                weights_np.std()
            ),
        }

        for role in [
            "train",
            "val",
            "test",
        ]:
            metrics = (
                metrics_by_role[
                    role
                ]
            )

            for field in [
                "accuracy",
                "wga",
                "loss",
                "worst_group_loss",
                "worst_class_accuracy",
                "mean_class_accuracy",
                "mean_group_accuracy",
                "worst_class_loss",
            ]:
                history_row[
                    f"{role}_{field}"
                ] = metrics[field]

        append_csv_rows(
            history_csv,
            [history_row],
        )

        log(
            (
                f"[epoch {epoch:03d}/{FINAL_EPOCHS}] "
                f"train_acc="
                f"{100 * metrics_by_role['train']['accuracy']:.2f}% "
                f"train_WGA="
                f"{100 * metrics_by_role['train']['wga']:.2f}% | "
                f"val_acc="
                f"{100 * metrics_by_role['val']['accuracy']:.2f}% "
                f"val_WGA="
                f"{100 * metrics_by_role['val']['wga']:.2f}% "
                f"val_worst_class="
                f"{100 * metrics_by_role['val']['worst_class_accuracy']:.2f}% | "
                f"test_acc="
                f"{100 * metrics_by_role['test']['accuracy']:.2f}% "
                f"test_WGA="
                f"{100 * metrics_by_role['test']['wga']:.2f}%"
            )
        )

        if (
            epoch == 1
            or epoch % EVAL_EVERY == 0
            or epoch == FINAL_EPOCHS
        ):
            save_trajectory_plots(
                metric_rows,
                group_rows,
                class_rows,
                plots_root,
            )

    # --------------------------------------------------------
    # Restore each validation-selected checkpoint and evaluate.
    # --------------------------------------------------------

    selection_results = []

    for rule_name, rule in SELECTION_RULES.items():
        selected_checkpoint = torch.load(
            selection_checkpoint_root
            / f"{rule_name}.pt",
            map_location="cpu",
            weights_only=False,
        )

        model.load_state_dict(
            selected_checkpoint[
                "model_sd"
            ]
        )
        model.to(DEVICE)
        model.eval()

        selected_metrics = {
            role: evaluate_classifier(
                model,
                dataset,
            )
            for role, dataset
            in datasets_by_role.items()
        }

        selected_epoch = int(
            selected_checkpoint["epoch"]
        )

        for role, metrics in selected_metrics.items():
            record_metrics(
                selected_epoch,
                role,
                metrics,
                phase="selected",
                selection_rule=rule_name,
            )

        compact = {
            "schema_version": 3,
            "stage": "final_classifier",
            "seed": seed,
            "split_seed": split_seed,
            "model_sd": (
                cpu_state_dict(model)
            ),
            "selection_rule": (
                rule_name
            ),
            "selection_field": (
                rule["field"]
            ),
            "selection_mode": (
                rule["mode"]
            ),
            "selected_epoch": (
                selected_epoch
            ),
            "selection_value": (
                selected_checkpoint[
                    "selection_value"
                ]
            ),
            "selected_metrics": (
                selected_metrics
            ),
            "weighting_policy": (
                WEIGHTING_POLICY
            ),
            "weighting_metadata": (
                weighting_metadata
            ),
            "embedding_cache_root": str(
                EMBEDDING_CACHE_ROOT
            ),
            "erm_fingerprint": (
                EXPECTED_ERM_FINGERPRINT
            ),
        }

        atomic_torch_save(
            compact,
            selected_model_root
            / f"{rule_name}.pt",
        )

        test_metrics = (
            selected_metrics["test"]
        )

        selection_results.append({
            "seed": seed,
            "split_seed": split_seed,
            "selection_rule": (
                rule_name
            ),
            "selection_label": (
                rule["label"]
            ),
            "selection_field": (
                rule["field"]
            ),
            "selection_mode": (
                rule["mode"]
            ),
            "selected_epoch": (
                selected_epoch
            ),
            "validation_selection_value": float(
                selected_checkpoint[
                    "selection_value"
                ]
            ),
            "test_accuracy": (
                test_metrics["accuracy"]
            ),
            "test_wga": (
                test_metrics["wga"]
            ),
            "test_mean_group_accuracy": (
                test_metrics[
                    "mean_group_accuracy"
                ]
            ),
            "test_loss": (
                test_metrics["loss"]
            ),
            "test_worst_group_loss": (
                test_metrics[
                    "worst_group_loss"
                ]
            ),
            "test_worst_class_accuracy": (
                test_metrics[
                    "worst_class_accuracy"
                ]
            ),
            "test_mean_class_accuracy": (
                test_metrics[
                    "mean_class_accuracy"
                ]
            ),
            "test_worst_class_loss": (
                test_metrics[
                    "worst_class_loss"
                ]
            ),
        })

    write_csv(
        seed_root
        / "selection_results.csv",
        selection_results,
    )

    # Compatibility export: primary = val_wga by default.
    primary_compact = torch.load(
        selected_model_root
        / (
            PRIMARY_SELECTION_RULE
            + ".pt"
        ),
        map_location="cpu",
        weights_only=False,
    )

    atomic_torch_save(
        primary_compact,
        seed_root
        / "best_final_classifier.pt",
    )

    oracle_results = []

    for (
        oracle_name,
        oracle_rule,
    ) in ORACLE_RULES.items():
        tracker = (
            oracle_trackers[
                oracle_name
            ]
        )

        test_metrics = (
            tracker["test_metrics"]
        )

        oracle_results.append({
            "seed": seed,
            "split_seed": split_seed,
            "oracle_rule": (
                oracle_name
            ),
            "oracle_label": (
                oracle_rule["label"]
            ),
            "oracle_field": (
                oracle_rule["field"]
            ),
            "oracle_mode": (
                oracle_rule["mode"]
            ),
            "oracle_epoch": int(
                tracker["best_epoch"]
            ),
            "oracle_value": float(
                tracker["best_value"]
            ),
            "test_accuracy": (
                test_metrics["accuracy"]
            ),
            "test_wga": (
                test_metrics["wga"]
            ),
            "test_loss": (
                test_metrics["loss"]
            ),
            "test_worst_group_loss": (
                test_metrics[
                    "worst_group_loss"
                ]
            ),
            "test_worst_class_accuracy": (
                test_metrics[
                    "worst_class_accuracy"
                ]
            ),
        })

    write_csv(
        seed_root
        / "oracle_results.csv",
        oracle_results,
    )

    save_selection_comparison_plot(
        selection_results,
        oracle_results,
        plots_root
        / "selection_rule_test_comparison.png",
    )

    seed_summary = {
        "seed": seed,
        "split_seed": split_seed,
        "selection_results": (
            selection_results
        ),
        "oracle_results": (
            oracle_results
        ),
        "initial_metrics": (
            initial_metrics
        ),
        "elapsed_seconds": (
            time.monotonic()
            - seed_start
        ),
        # Compute cost of this seed (training + all evaluations).
        "peak_gpu_memory_bytes": (
            int(torch.cuda.max_memory_allocated())
            if torch.cuda.is_available()
            else None
        ),
        "gpu_name": (
            torch.cuda.get_device_name()
            if torch.cuda.is_available()
            else None
        ),
        "seed_root": str(
            seed_root
        ),
    }

    write_json(
        seed_root
        / "summary.json",
        seed_summary,
    )

    with open(
        seed_root
        / "summary.txt",
        "w",
        encoding="utf-8",
    ) as fout:
        fout.write(
            f"FINAL CLASSIFIER SEED {seed}\n"
        )
        fout.write(
            "=" * 132
            + "\n"
        )
        fout.write(
            f"Split seed: {split_seed}\n"
        )
        fout.write(
            f"Weighting policy: {WEIGHTING_POLICY}\n"
        )
        fout.write(
            f"Weighting metadata: {json.dumps(json_value(weighting_metadata), sort_keys=True)}\n\n"
        )

        fout.write(
            "VALIDATION SELECTION -> TEST RESULTS\n"
        )
        fout.write(
            "-" * 132
            + "\n"
        )
        fout.write(
            (
                f"{'selection_rule':30s} "
                f"{'epoch':>6s} "
                f"{'val criterion':>14s} "
                f"{'test acc':>12s} "
                f"{'test WGA':>12s} "
                f"{'test worst-class':>17s} "
                f"{'test loss':>12s}\n"
            )
        )

        for row in selection_results:
            fout.write(
                (
                    f"{row['selection_rule']:30s} "
                    f"{row['selected_epoch']:6d} "
                    f"{row['validation_selection_value']:14.6f} "
                    f"{row['test_accuracy']:12.6f} "
                    f"{row['test_wga']:12.6f} "
                    f"{row['test_worst_class_accuracy']:17.6f} "
                    f"{row['test_loss']:12.6f}\n"
                )
            )

        fout.write(
            "\nTEST ORACLES -- DIAGNOSTIC ONLY\n"
        )
        fout.write(
            "-" * 132
            + "\n"
        )

        for row in oracle_results:
            fout.write(
                (
                    f"{row['oracle_rule']:30s} "
                    f"epoch={row['oracle_epoch']:3d} "
                    f"test_acc={row['test_accuracy']:.6f} "
                    f"test_WGA={row['test_wga']:.6f} "
                    f"worst_class="
                    f"{row['test_worst_class_accuracy']:.6f}\n"
                )
            )

    log("")
    log(
        f"Seed {seed} complete."
    )

    for row in selection_results:
        log(
            (
                f"[{row['selection_rule']}] "
                f"epoch={row['selected_epoch']} "
                f"test_acc="
                f"{100 * row['test_accuracy']:.2f}% "
                f"test_WGA="
                f"{100 * row['test_wga']:.2f}%"
            )
        )

    return seed_summary

In [9]:
# ============================================================
# 9. Run all requested final-classifier seeds
# ============================================================

ALL_SEED_SUMMARIES = []

for seed in FINAL_CLASSIFIER_SEEDS:
    print(
        "\n"
        + "=" * 110
    )
    print(
        f"STARTING FINAL CLASSIFIER SEED {seed}"
    )
    print(
        "=" * 110
    )

    summary = run_one_seed(
        seed
    )

    ALL_SEED_SUMMARIES.append(
        summary
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(
    "\nAll final-classifier seeds finished."
)

print(
    "Experiment root:",
    EXPERIMENT_ROOT,
)


STARTING FINAL CLASSIFIER SEED 0
FINAL CLASSIFIER SEED 0
data split seed = 0
weighting policy = fixed_random_uniform_per_sample
train=val_subset1: n=719
val=val_subset2: n=480
test=test: n=5794


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[epoch 001/50] train_acc=91.24% train_WGA=84.70% | val_acc=90.83% val_WGA=81.62% val_worst_class=90.26% | test_acc=91.04% test_WGA=82.79%
[epoch 002/50] train_acc=94.71% train_WGA=74.36% | val_acc=94.58% val_WGA=67.27% val_worst_class=79.00% | test_acc=95.24% test_WGA=78.97%
[epoch 003/50] train_acc=94.71% train_WGA=87.18% | val_acc=95.42% val_WGA=81.82% val_worst_class=89.00% | test_acc=95.50% test_WGA=87.69%
[epoch 004/50] train_acc=95.55% train_WGA=88.46% | val_acc=95.83% val_WGA=83.64% val_worst_class=90.00% | test_acc=95.65% test_WGA=87.38%
[epoch 005/50] train_acc=96.52% train_WGA=89.77% | val_acc=95.62% val_WGA=83.64% val_worst_class=89.00% | test_acc=95.77% test_WGA=87.23%
[epoch 006/50] train_acc=96.52% train_WGA=94.32% | val_acc=94.38% val_WGA=85.45% val_worst_class=91.00% | test_acc=94.86% test_WGA=90.03%
[epoch 007/50] train_acc=97.08% train_WGA=89.77% | val_acc=94.17% val_WGA=80.00% val_worst_class=85.00% | test_acc=95.18% test_WGA=86.60%
[epoch 008/50] train_acc=97.77% tr

In [10]:
# ============================================================
# 10. Aggregate all seeds + write root summary.txt
# ============================================================

all_selection_results = []
all_oracle_results = []

for seed_summary in ALL_SEED_SUMMARIES:
    all_selection_results.extend(
        seed_summary[
            "selection_results"
        ]
    )
    all_oracle_results.extend(
        seed_summary[
            "oracle_results"
        ]
    )

write_csv(
    EXPERIMENT_ROOT
    / "selection_results_all_seeds.csv",
    all_selection_results,
)

write_csv(
    EXPERIMENT_ROOT
    / "oracle_results_all_seeds.csv",
    all_oracle_results,
)


# ------------------------------------------------------------
# Mean +/- SD across seeds for every validation selection rule.
# ------------------------------------------------------------

aggregate_selection_rows = []

for rule_name in SELECTION_RULES:
    rows = [
        row
        for row in all_selection_results
        if row[
            "selection_rule"
        ] == rule_name
    ]

    aggregate = {
        "selection_rule": rule_name,
        "selection_label": (
            SELECTION_RULES[
                rule_name
            ][
                "label"
            ]
        ),
        "n_seeds": len(rows),
    }

    for field in [
        "selected_epoch",
        "test_accuracy",
        "test_wga",
        "test_mean_group_accuracy",
        "test_loss",
        "test_worst_group_loss",
        "test_worst_class_accuracy",
        "test_mean_class_accuracy",
        "test_worst_class_loss",
    ]:
        mean_value, std_value = (
            mean_std(
                [
                    row[field]
                    for row in rows
                ]
            )
        )

        aggregate[
            field
            + "_mean"
        ] = mean_value

        aggregate[
            field
            + "_std"
        ] = std_value

    aggregate_selection_rows.append(
        aggregate
    )

write_csv(
    EXPERIMENT_ROOT
    / "selection_metrics_mean_std.csv",
    aggregate_selection_rows,
)


# ------------------------------------------------------------
# Aggregate the diagnostic test oracles.
# ------------------------------------------------------------

aggregate_oracle_rows = []

for oracle_name in ORACLE_RULES:
    rows = [
        row
        for row in all_oracle_results
        if row[
            "oracle_rule"
        ] == oracle_name
    ]

    aggregate = {
        "oracle_rule": oracle_name,
        "oracle_label": (
            ORACLE_RULES[
                oracle_name
            ][
                "label"
            ]
        ),
        "n_seeds": len(rows),
    }

    for field in [
        "oracle_epoch",
        "test_accuracy",
        "test_wga",
        "test_loss",
        "test_worst_group_loss",
        "test_worst_class_accuracy",
    ]:
        mean_value, std_value = (
            mean_std(
                [
                    row[field]
                    for row in rows
                ]
            )
        )

        aggregate[
            field
            + "_mean"
        ] = mean_value

        aggregate[
            field
            + "_std"
        ] = std_value

    aggregate_oracle_rows.append(
        aggregate
    )

write_csv(
    EXPERIMENT_ROOT
    / "oracle_metrics_mean_std.csv",
    aggregate_oracle_rows,
)


# ------------------------------------------------------------
# Aggregate selection-rule test plot.
# ------------------------------------------------------------

labels = [
    row[
        "selection_rule"
    ]
    for row in aggregate_selection_rows
]

fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 10),
)

for ax, field, ylabel in [
    (
        axes[0],
        "test_accuracy",
        "Test overall accuracy",
    ),
    (
        axes[1],
        "test_wga",
        "Test WGA",
    ),
]:
    means = np.asarray(
        [
            row[
                field
                + "_mean"
            ]
            for row in aggregate_selection_rows
        ]
    )

    stds = np.asarray(
        [
            row[
                field
                + "_std"
            ]
            for row in aggregate_selection_rows
        ]
    )

    ax.bar(
        labels,
        means,
        yerr=stds,
        capsize=4,
    )

    ax.set_ylabel(
        ylabel
    )

    ax.tick_params(
        axis="x",
        labelrotation=35,
    )

    ax.grid(
        True,
        axis="y",
        alpha=0.25,
    )

fig.suptitle(
    "Validation selection rules: test result mean ± seed SD"
)

fig.tight_layout()

fig.savefig(
    AGGREGATE_PLOTS_ROOT
    / "selection_rules_test_mean_std.png",
    dpi=170,
    bbox_inches="tight",
)

plt.close(fig)


# ------------------------------------------------------------
# Multi-seed trajectory aggregation.
# ------------------------------------------------------------

trajectory_rows = []

for seed in FINAL_CLASSIFIER_SEEDS:
    path = (
        EXPERIMENT_ROOT
        / f"seed_{seed}"
        / "plots"
        / "data"
        / "metrics.csv"
    )

    with open(
        path,
        newline="",
        encoding="utf-8",
    ) as fin:
        rows = list(
            csv.DictReader(fin)
        )

    trajectory_rows.extend(
        [
            row
            for row in rows
            if row[
                "phase"
            ] == "trajectory"
        ]
    )

trajectory_grouped = defaultdict(list)

for row in trajectory_rows:
    key = (
        row["role"],
        int(
            row["epoch"]
        ),
    )
    trajectory_grouped[
        key
    ].append(row)

trajectory_fields = [
    "accuracy",
    "wga",
    "loss",
    "worst_group_loss",
    "worst_class_accuracy",
    "mean_class_accuracy",
    "mean_group_accuracy",
    "worst_class_loss",
]

trajectory_aggregate = []

for (
    role,
    epoch,
), rows in sorted(
    trajectory_grouped.items()
):
    item = {
        "role": role,
        "epoch": epoch,
        "n_seeds": len(rows),
    }

    for field in trajectory_fields:
        mean_value, std_value = mean_std(
            [
                float(
                    row[field]
                )
                for row in rows
            ]
        )

        item[
            field
            + "_mean"
        ] = mean_value

        item[
            field
            + "_std"
        ] = std_value

    trajectory_aggregate.append(
        item
    )

write_csv(
    EXPERIMENT_ROOT
    / "trajectory_metrics_mean_std.csv",
    trajectory_aggregate,
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 9),
)

for ax, field, ylabel in [
    (
        axes[0, 0],
        "accuracy",
        "Accuracy",
    ),
    (
        axes[0, 1],
        "wga",
        "WGA",
    ),
    (
        axes[1, 0],
        "loss",
        "Mean CE",
    ),
    (
        axes[1, 1],
        "worst_group_loss",
        "Worst-group mean CE",
    ),
]:
    for role in [
        "train",
        "val",
        "test",
    ]:
        rows = sorted(
            [
                row
                for row in trajectory_aggregate
                if row[
                    "role"
                ] == role
            ],
            key=lambda row: row["epoch"],
        )

        x = np.asarray(
            [
                row["epoch"]
                for row in rows
            ]
        )
        mean = np.asarray(
            [
                row[
                    field
                    + "_mean"
                ]
                for row in rows
            ]
        )
        std = np.asarray(
            [
                row[
                    field
                    + "_std"
                ]
                for row in rows
            ]
        )

        ax.plot(
            x,
            mean,
            label=role,
        )

        ax.fill_between(
            x,
            mean - std,
            mean + std,
            alpha=0.15,
        )

    ax.set_xlabel(
        "Epoch"
    )
    ax.set_ylabel(
        ylabel
    )
    ax.grid(
        True,
        alpha=0.25,
    )
    ax.legend()

fig.suptitle(
    "Multi-seed final-classifier trajectory: mean ± seed SD"
)

fig.tight_layout()

fig.savefig(
    AGGREGATE_PLOTS_ROOT
    / "trajectory_mean_std.png",
    dpi=170,
    bbox_inches="tight",
)

plt.close(fig)


# ------------------------------------------------------------
# Root summary files BESIDE the per-seed folders.
# ------------------------------------------------------------

root_summary = {
    "status": (
        "complete_before_gradcam"
    ),
    "experiment_root": str(
        EXPERIMENT_ROOT
    ),
    "seeds": FINAL_CLASSIFIER_SEEDS,
    "selection_results": (
        all_selection_results
    ),
    "selection_metrics_mean_std": (
        aggregate_selection_rows
    ),
    "oracle_results": (
        all_oracle_results
    ),
    "oracle_metrics_mean_std": (
        aggregate_oracle_rows
    ),
    "config": root_config,
}

write_json(
    EXPERIMENT_ROOT
    / "summary.json",
    root_summary,
)

with open(
    EXPERIMENT_ROOT
    / "summary.txt",
    "w",
    encoding="utf-8",
) as fout:
    fout.write(
        "MULTI-SEED FINAL CLASSIFIER SUMMARY\n"
    )
    fout.write(
        "=" * 150
        + "\n"
    )
    fout.write(
        f"Seeds: {FINAL_CLASSIFIER_SEEDS}\n"
    )
    fout.write(
        f"Weighting policy: {WEIGHTING_POLICY}\n"
    )
    fout.write(
        f"Weighting signature: {json.dumps(json_value(WEIGHTING_SIGNATURE), sort_keys=True)}\n"
    )
    fout.write(
        f"Split_val: {SPLIT_VAL}\n"
    )
    fout.write(
        f"Vary data split with seed: "
        f"{VARY_DATA_SPLIT_WITH_SEED}\n"
    )
    fout.write(
        f"Fixed data split seed: "
        f"{DATA_SPLIT_SEED}\n\n"
    )

    for seed in FINAL_CLASSIFIER_SEEDS:
        fout.write(
            f"\nSEED {seed}\n"
        )
        fout.write(
            "-" * 150
            + "\n"
        )
        fout.write(
            (
                f"{'selection_rule':30s} "
                f"{'epoch':>6s} "
                f"{'test acc':>12s} "
                f"{'test WGA':>12s} "
                f"{'test worst-class':>17s} "
                f"{'test loss':>12s}\n"
            )
        )

        rows = [
            row
            for row in all_selection_results
            if row["seed"] == seed
        ]

        for row in rows:
            fout.write(
                (
                    f"{row['selection_rule']:30s} "
                    f"{row['selected_epoch']:6d} "
                    f"{row['test_accuracy']:12.6f} "
                    f"{row['test_wga']:12.6f} "
                    f"{row['test_worst_class_accuracy']:17.6f} "
                    f"{row['test_loss']:12.6f}\n"
                )
            )

        fout.write(
            "\n  TEST ORACLES -- DIAGNOSTIC ONLY\n"
        )

        for row in [
            row
            for row in all_oracle_results
            if row["seed"] == seed
        ]:
            fout.write(
                (
                    f"  {row['oracle_rule']:28s} "
                    f"epoch={row['oracle_epoch']:3d} "
                    f"test_acc={row['test_accuracy']:.6f} "
                    f"test_WGA={row['test_wga']:.6f}\n"
                )
            )

    fout.write(
        "\n\nAGGREGATE ACROSS SEEDS\n"
    )
    fout.write(
        "=" * 150
        + "\n"
    )
    fout.write(
        (
            f"{'selection_rule':30s} "
            f"{'test acc mean±sd':>25s} "
            f"{'test WGA mean±sd':>25s} "
            f"{'worst-class mean±sd':>25s}\n"
        )
    )

    for row in aggregate_selection_rows:
        fout.write(
            (
                f"{row['selection_rule']:30s} "
                f"{row['test_accuracy_mean']:.6f}±"
                f"{row['test_accuracy_std']:.6f} "
                f"{row['test_wga_mean']:12.6f}±"
                f"{row['test_wga_std']:.6f} "
                f"{row['test_worst_class_accuracy_mean']:12.6f}±"
                f"{row['test_worst_class_accuracy_std']:.6f}\n"
            )
        )

    fout.write(
        "\nAGGREGATE TEST ORACLES -- DIAGNOSTIC ONLY\n"
    )
    fout.write(
        "-" * 150
        + "\n"
    )

    for row in aggregate_oracle_rows:
        fout.write(
            (
                f"{row['oracle_rule']:30s} "
                f"test_acc={row['test_accuracy_mean']:.6f}±"
                f"{row['test_accuracy_std']:.6f} "
                f"test_WGA={row['test_wga_mean']:.6f}±"
                f"{row['test_wga_std']:.6f}\n"
            )
        )

print(
    "Root multi-seed summary written to:"
)
print(
    EXPERIMENT_ROOT
    / "summary.txt"
)

Root multi-seed summary written to:
/content/drive/MyDrive/evidential_alignment_colab/fixed_random_weight_final_multiseed_multiselection/random_uniform_0.0_1.0_lr_0.01_splitval_0.6_20261013_140627/summary.txt


In [11]:
# ============================================================
# 11. Optional Grad-CAM for selected rule(s) across all seeds
# ============================================================

def prepare_gradcam_repo_and_data():
    if not REPO_DIR.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/TheRealMmd/evidential_alignment.git",
                str(REPO_DIR),
            ],
            check=True,
        )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "timm",
            "transformers",
            "wilds",
            "wget",
        ],
        check=True,
    )

    DATA_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )
    WB_ARCHIVE.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if not (
        WB_ARCHIVE.exists()
        and WB_ARCHIVE.stat().st_size
        > 100_000_000
    ):
        subprocess.run(
            [
                "wget",
                "-c",
                "https://downloads.cs.stanford.edu/nlp/data/dro/"
                "waterbird_complete95_forest2water2.tar.gz",
                "-O",
                str(WB_ARCHIVE),
            ],
            check=True,
        )

    if not (
        WB_DIR
        / "metadata.csv"
    ).exists():
        subprocess.run(
            [
                "tar",
                "-xzf",
                str(WB_ARCHIVE),
                "-C",
                str(DATA_ROOT),
            ],
            check=True,
        )

    config_path = (
        REPO_DIR
        / "config.py"
    )

    config_text = (
        config_path.read_text(
            encoding="utf-8"
        )
    )

    placeholder = (
        '"waterbirds": "/path/to/datasets/waterbirds"'
    )

    if placeholder in config_text:
        config_text = config_text.replace(
            placeholder,
            f'"waterbirds": "{WB_DIR}"',
        )
    else:
        config_text, count = re.subn(
            r'("waterbirds"\s*:\s*)["\'][^"\']*["\']',
            lambda match: (
                match.group(1)
                + '"'
                + str(WB_DIR)
                + '"'
            ),
            config_text,
            count=1,
        )

        if count != 1:
            raise RuntimeError(
                "Could not patch Waterbirds path in config.py"
            )

    config_path.write_text(
        config_text,
        encoding="utf-8",
    )


def denormalize_image(
    x,
    dataset,
):
    display = (
        x.detach()
        .cpu()[0]
        .clone()
    )

    transform_list = list(
        getattr(
            getattr(
                dataset,
                "transform",
                None,
            ),
            "transforms",
            [],
        )
    )

    for transform in reversed(
        transform_list
    ):
        if (
            hasattr(
                transform,
                "mean",
            )
            and hasattr(
                transform,
                "std",
            )
        ):
            mean = torch.as_tensor(
                transform.mean,
                dtype=display.dtype,
            )[:, None, None]

            std = torch.as_tensor(
                transform.std,
                dtype=display.dtype,
            )[:, None, None]

            display = (
                display
                * std
                + mean
            )
            break

    return (
        display.clamp(
            0,
            1,
        )
        .permute(
            1,
            2,
            0,
        )
        .numpy()
    )


def run_gradcam_all_seeds():
    if not ENABLE_GRADCAM:
        print(
            "Grad-CAM disabled."
        )
        return []

    prepare_gradcam_repo_and_data()

    if str(REPO_DIR) not in sys.path:
        sys.path.insert(
            0,
            str(REPO_DIR),
        )

    import data
    from data.data_utils import prepare_data
    from models.classifier import Classifier

    image_config = SimpleNamespace(
        dataset="waterbirds",
        resolution=224,
        split_train=1.0,
        split_val=float(
            SPLIT_VAL
        ),
        seed=int(
            DATA_SPLIT_SEED
        ),
        train_split="train",
        batch_size=int(
            BATCH_SIZE
        ),
        num_workers=0,
    )

    image_datasets, _ = prepare_data(
        image_config
    )

    if GRADCAM_SPLIT not in image_datasets:
        raise ValueError(
            f"Grad-CAM split unavailable: "
            f"{GRADCAM_SPLIT}"
        )

    image_dataset = image_datasets[
        GRADCAM_SPLIT
    ]

    # This control experiment has no Rater checkpoint.
    # Rebuild the image-space predictor from the same saved Waterbirds-ERM
    # backbone that produced the frozen embedding cache.
    erm_model_path = (
        DEFAULT_ERM_MODEL
    )

    if not erm_model_path.exists():
        raise FileNotFoundError(
            erm_model_path
        )

    feature_model = Classifier(
        backbone="resnet50",
        num_classes=2,
        pretrained=False,
    )

    erm_checkpoint = torch.load(
        erm_model_path,
        map_location="cpu",
        weights_only=False,
    )

    erm_state = (
        erm_checkpoint["model_sd"]
        if (
            isinstance(
                erm_checkpoint,
                dict,
            )
            and "model_sd"
            in erm_checkpoint
        )
        else erm_checkpoint
    )

    erm_state = {
        key: value
        for key, value
        in erm_state.items()
        if not key.endswith(
            "position_ids"
        )
    }

    feature_model.load_state_dict(
        erm_state,
        strict=False,
    )

    backbone = (
        feature_model.backbone
        .cpu()
        .eval()
    )

    target_layer = getattr(
        backbone,
        "layer4",
        None,
    )

    if target_layer is None:
        raise RuntimeError(
            "ResNet backbone has no layer4."
        )

    groups = np.asarray(
        image_dataset.group_array
    )

    sample_indices = []

    for gid in sorted(
        np.unique(groups)
    ):
        candidates = np.flatnonzero(
            groups == gid
        )

        sample_indices.extend(
            [
                int(index)
                for index in candidates[
                    :GRADCAM_SAMPLES_PER_GROUP
                ]
            ]
        )

    all_manifest = []

    for seed in FINAL_CLASSIFIER_SEEDS:
        for rule_name in GRADCAM_SELECTION_RULES:
            compact_path = (
                EXPERIMENT_ROOT
                / f"seed_{seed}"
                / "selected_models"
                / f"{rule_name}.pt"
            )

            compact = torch.load(
                compact_path,
                map_location="cpu",
                weights_only=False,
            )

            head = FinalLinearClassifier(
                2048,
                2,
            )

            head.load_state_dict(
                compact[
                    "model_sd"
                ]
            )
            head.cpu()
            head.eval()

            out_dir = (
                EXPERIMENT_ROOT
                / f"seed_{seed}"
                / "plots"
                / "gradcam"
                / rule_name
            )

            out_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

            activations = []

            hook = (
                target_layer
                .register_forward_hook(
                    lambda module, args, output: (
                        activations.append(
                            output
                        )
                    )
                )
            )

            manifest = []

            try:
                for index in sample_indices:
                    (
                        x,
                        y,
                        group,
                        attr,
                    ) = image_dataset[
                        index
                    ]

                    x = (
                        x.detach()
                        .cpu()
                        .unsqueeze(0)
                        .requires_grad_(True)
                    )

                    activations.clear()

                    with torch.enable_grad():
                        embedding = backbone(
                            x
                        )

                        activation = (
                            activations[-1]
                        )

                        logits = head(
                            embedding
                        )

                        probabilities = (
                            torch.softmax(
                                logits,
                                dim=1,
                            )
                        )

                        pred = int(
                            logits.argmax(
                                dim=1
                            ).item()
                        )

                        target = logits[
                            0,
                            pred,
                        ]

                        gradients = (
                            torch.autograd.grad(
                                target,
                                activation,
                            )[0]
                        )

                        channel_weights = (
                            gradients.mean(
                                dim=(
                                    2,
                                    3,
                                ),
                                keepdim=True,
                            )
                        )

                        cam = F.relu(
                            (
                                channel_weights
                                * activation
                            ).sum(
                                dim=1
                            )
                        )

                        cam = F.interpolate(
                            cam[:, None],
                            size=x.shape[-2:],
                            mode="bilinear",
                            align_corners=False,
                        )[0, 0]

                        cam = cam.detach()

                        cam = (
                            cam
                            - cam.min()
                        ) / (
                            cam.max()
                            - cam.min()
                        ).clamp_min(
                            1e-12
                        )

                        cam_np = (
                            cam.numpy()
                        )

                    display = (
                        denormalize_image(
                            x,
                            image_dataset,
                        )
                    )

                    stem = (
                        f"{GRADCAM_SPLIT}_"
                        f"g{int(group)}_"
                        f"sample{index:06d}"
                    )

                    fig, axes = plt.subplots(
                        1,
                        2,
                        figsize=(9, 4),
                    )

                    axes[0].imshow(
                        display
                    )
                    axes[0].set_title(
                        (
                            f"Original\n"
                            f"y={int(y)}, "
                            f"group={int(group)}"
                        )
                    )

                    axes[1].imshow(
                        display
                    )
                    axes[1].imshow(
                        cam_np,
                        alpha=0.45,
                    )
                    axes[1].set_title(
                        (
                            f"{rule_name}\n"
                            f"pred={pred}, "
                            f"p="
                            f"{float(probabilities[0, pred]):.3f}"
                        )
                    )

                    for ax in axes:
                        ax.axis(
                            "off"
                        )

                    fig.tight_layout()

                    fig.savefig(
                        out_dir
                        / f"{stem}.png",
                        dpi=150,
                        bbox_inches="tight",
                    )

                    plt.close(fig)

                    np.savez_compressed(
                        out_dir
                        / f"{stem}.npz",
                        image=display,
                        cam=cam_np,
                        logits=(
                            logits.detach()
                            .numpy()
                        ),
                        probabilities=(
                            probabilities.detach()
                            .numpy()
                        ),
                    )

                    manifest.append({
                        "seed": seed,
                        "selection_rule": (
                            rule_name
                        ),
                        "split": (
                            GRADCAM_SPLIT
                        ),
                        "sample_index": (
                            index
                        ),
                        "label": int(y),
                        "group": int(group),
                        "attribute": int(attr),
                        "predicted_class": (
                            pred
                        ),
                        "predicted_probability": float(
                            probabilities[
                                0,
                                pred,
                            ].detach().item()
                        ),
                        "artifact": stem,
                    })

            finally:
                hook.remove()
                activations.clear()

            write_json(
                out_dir
                / "samples.json",
                manifest,
            )

            all_manifest.extend(
                manifest
            )

    return all_manifest


GRADCAM_MANIFEST = (
    run_gradcam_all_seeds()
)

summary_path = (
    EXPERIMENT_ROOT
    / "summary.json"
)

root_summary = json.loads(
    summary_path.read_text(
        encoding="utf-8"
    )
)

root_summary["status"] = "complete"
root_summary[
    "gradcam_samples_saved"
] = len(
    GRADCAM_MANIFEST
)

write_json(
    summary_path,
    root_summary,
)

print(
    "Grad-CAM artifacts saved:",
    len(
        GRADCAM_MANIFEST
    ),
)

/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


train: 4795 (40.68%)
Group0 (y=0, a=0): 3498 (72.95%)
Group1 (y=0, a=1): 184 (3.84%)
Group2 (y=1, a=0): 56 (1.17%)
Group3 (y=1, a=1): 1057 (22.04%)

train: 4795 (40.68%)
Group0 (y=0, a=0): 3498 (72.95%)
Group1 (y=0, a=1): 184 (3.84%)
Group2 (y=1, a=0): 56 (1.17%)
Group3 (y=1, a=1): 1057 (22.04%)

val: 1199 (10.17%)
Group0 (y=0, a=0): 467 (38.95%)
Group1 (y=0, a=1): 466 (38.87%)
Group2 (y=1, a=0): 133 (11.09%)
Group3 (y=1, a=1): 133 (11.09%)

test: 5794 (49.15%)
Group0 (y=0, a=0): 2255 (38.92%)
Group1 (y=0, a=1): 2255 (38.92%)
Group2 (y=1, a=0): 642 (11.08%)
Group3 (y=1, a=1): 642 (11.08%)

val (59.97%): 719 (59.97%)
Group0 (y=0, a=0): 280 (38.94%)
Group1 (y=0, a=1): 278 (38.66%)
Group2 (y=1, a=0): 87 (12.10%)
Group3 (y=1, a=1): 74 (10.29%)

val (40.03%): 480 (40.03%)
Group0 (y=0, a=0): 187 (38.96%)
Group1 (y=0, a=1): 188 (39.17%)
Group2 (y=1, a=0): 46 (9.58%)
Group3 (y=1, a=1): 59 (12.29%)

train: 4795 (40.68%)
Group0 (y=0, a=0): 3498 (72.95%)
Group1 (y=0, a=1): 184 (3.84%)
Group2 (y=1

/tmp/ipykernel_2867/2818976420.py:523: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  f"{float(probabilities[0, pred]):.3f}"


Grad-CAM artifacts saved: 28


## Output and reporting

This notebook uses **no Rater**. Its weighting policy is:

> each training sample receives one fixed Uniform[0,1) weight per seed.

Everything else mirrors the multi-seed final-classifier pipeline: the same
embeddings, fresh `Linear(2048,2)` classifier, optimizer, 50 epochs, validation
selection rules, test oracles, metrics, checkpoints, multi-seed summaries, and
Grad-CAM.

For each seed the exact fixed sample weights are saved as:

```text
seed_<s>/fixed_training_weights.pt
seed_<s>/fixed_training_weights.csv
```

so the experiment is fully reproducible and auditable.

The root `summary.txt` sits beside the seed folders and reports test accuracy
and test WGA under every validation selection rule, plus diagnostic test
oracles and mean ± SD across seeds.

Root Drive namespace:

```text
fixed_random_weight_final_multiseed_multiselection/
└── ONE_EXPERIMENT_FOLDER/
    ├── summary.txt
    ├── summary.json
    ├── selection_results_all_seeds.csv
    ├── selection_metrics_mean_std.csv
    ├── oracle_results_all_seeds.csv
    ├── oracle_metrics_mean_std.csv
    ├── trajectory_metrics_mean_std.csv
    ├── aggregate_plots/
    ├── seed_0/
    ├── seed_1/
    └── seed_2/
```

For paper reporting, use the validation-selected results. The test-oracle
results are diagnostics only.